In [1]:
pip install os

ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
import random
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin

print("Imports correctos")

Imports correctos


In [8]:
import os
import time
import requests
import pandas as pd


# ============================================================
# CONFIGURACIÓN
# ============================================================

SITE_ID = "MLM"

BUSQUEDA = "laptops"

LIMIT_POR_PAGINA = 50

MAX_PRODUCTOS = 200

OUTPUT_DIR = "datos_crudos"

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "laptops_scraping_crudo.csv"
)


# ============================================================
# CONFIGURACIÓN HTTP
# ============================================================

BASE_URL = "https://api.mercadolibre.com"

HEADERS = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()

session.headers.update(HEADERS)


# ============================================================
# FUNCIÓN GENÉRICA PARA LLAMAR A LA API
# ============================================================

def hacer_request(url, params=None):

    try:

        response = session.get(
            url,
            params=params,
            timeout=30
        )

        print(
            f"GET {response.url}"
        )

        print(
            f"Status: {response.status_code}"
        )

        if response.status_code != 200:

            print(
                "Respuesta de error:"
            )

            print(
                response.text[:1000]
            )

            return None

        return response.json()

    except requests.exceptions.RequestException as e:

        print(
            f"❌ Error HTTP: {e}"
        )

        return None

    except ValueError as e:

        print(
            f"❌ Error interpretando JSON: {e}"
        )

        return None


# ============================================================
# BUSCAR LAPTOPS
# ============================================================

def buscar_laptops(
    query,
    max_productos=200
):

    url = (
        f"{BASE_URL}/sites/"
        f"{SITE_ID}/search"
    )

    productos = []

    offset = 0

    while len(productos) < max_productos:

        print("\n" + "=" * 60)

        print(
            f"Buscando productos "
            f"{offset + 1} - "
            f"{offset + LIMIT_POR_PAGINA}"
        )

        print("=" * 60)

        params = {

            "q": query,

            "limit": LIMIT_POR_PAGINA,

            "offset": offset
        }

        data = hacer_request(
            url,
            params=params
        )

        if data is None:

            print(
                "❌ No se pudo obtener "
                "la página."
            )

            break

        resultados = data.get(
            "results",
            []
        )

        if not resultados:

            print(
                "No hay más resultados."
            )

            break

        productos.extend(
            resultados
        )

        paging = data.get(
            "paging",
            {}
        )

        total = paging.get(
            "total",
            0
        )

        print(
            f"Total disponible: {total}"
        )

        print(
            f"Productos acumulados: "
            f"{len(productos)}"
        )

        offset += LIMIT_POR_PAGINA

        if offset >= total:

            break

        # Pausa entre solicitudes
        time.sleep(1)

    return productos[
        :max_productos
    ]


# ============================================================
# OBTENER INFORMACIÓN DETALLADA DEL ITEM
# ============================================================

def obtener_item(item_id):

    url = (
        f"{BASE_URL}/items/"
        f"{item_id}"
    )

    return hacer_request(url)


# ============================================================
# CONVERTIR ATRIBUTOS A DICCIONARIO
# ============================================================

def convertir_atributos(
    atributos
):

    resultado = {}

    if not atributos:

        return resultado

    for atributo in atributos:

        atributo_id = atributo.get(
            "id"
        )

        nombre = atributo.get(
            "name"
        )

        valor = atributo.get(
            "value_name"
        )

        if valor is None:

            valor = atributo.get(
                "value_id"
            )

        # Preferimos el ID porque
        # suele ser más estable.

        clave = (
            atributo_id
            or nombre
        )

        if clave:

            resultado[clave] = valor

        # También guardamos el nombre
        # legible cuando es diferente.

        if (
            nombre
            and nombre != clave
        ):

            resultado[
                nombre
            ] = valor

    return resultado


# ============================================================
# PROCESAR PRODUCTO
# ============================================================

def procesar_producto(
    resultado_busqueda
):

    item_id = resultado_busqueda.get(
        "id"
    )

    if not item_id:

        return None

    print("\n" + "-" * 60)

    print(
        f"Procesando: {item_id}"
    )

    print(
        resultado_busqueda.get(
            "title"
        )
    )

    # --------------------------------------------------------
    # Información detallada
    # --------------------------------------------------------

    item = obtener_item(
        item_id
    )

    if item is None:

        return None

    # --------------------------------------------------------
    # Información básica
    # --------------------------------------------------------

    registro = {

        "item_id":
            item.get("id"),

        "titulo":
            item.get("title"),

        "precio":
            item.get("price"),

        "moneda":
            item.get("currency_id"),

        "precio_original":
            item.get(
                "original_price"
            ),

        "categoria_id":
            item.get(
                "category_id"
            ),

        "condicion":
            item.get(
                "condition"
            ),

        "permalink":
            item.get(
                "permalink"
            ),

        "thumbnail":
            item.get(
                "thumbnail"
            ),

        "sold_quantity":
            item.get(
                "sold_quantity"
            ),

        "available_quantity":
            item.get(
                "available_quantity"
            )
    }

    # --------------------------------------------------------
    # Imágenes
    # --------------------------------------------------------

    pictures = item.get(
        "pictures",
        []
    )

    if pictures:

        registro[
            "imagen_principal"
        ] = pictures[0].get(
            "url"
        )

    else:

        registro[
            "imagen_principal"
        ] = None

    # --------------------------------------------------------
    # Atributos técnicos
    # --------------------------------------------------------

    atributos = convertir_atributos(
        item.get(
            "attributes",
            []
        )
    )

    registro.update(
        atributos
    )

    return registro


# ============================================================
# MAIN
# ============================================================

def main():

    print("=" * 60)

    print(
        "SCRAPER DE LAPTOPS "
        "MERCADO LIBRE"
    )

    print("=" * 60)

    # --------------------------------------------------------
    # 1. Buscar publicaciones
    # --------------------------------------------------------

    resultados = buscar_laptops(
        BUSQUEDA,
        MAX_PRODUCTOS
    )

    if not resultados:

        print(
            "\n❌ No se encontraron "
            "productos."
        )

        return

    print(
        f"\nSe encontraron "
        f"{len(resultados)} "
        f"publicaciones."
    )

    # --------------------------------------------------------
    # 2. Obtener detalles
    # --------------------------------------------------------

    registros = []

    for i, resultado in enumerate(
        resultados,
        start=1
    ):

        print(
            f"\nProducto "
            f"{i}/{len(resultados)}"
        )

        registro = (
            procesar_producto(
                resultado
            )
        )

        if registro:

            registros.append(
                registro
            )

        # Pausa entre solicitudes

        time.sleep(0.5)

    # --------------------------------------------------------
    # 3. DataFrame
    # --------------------------------------------------------

    if not registros:

        print(
            "\n❌ No se pudieron "
            "procesar productos."
        )

        return

    df = pd.DataFrame(
        registros
    )

    # --------------------------------------------------------
    # 4. Crear directorio
    # --------------------------------------------------------

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    # --------------------------------------------------------
    # 5. Guardar CSV
    # --------------------------------------------------------

    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # 6. Resumen
    # --------------------------------------------------------

    print("\n" + "=" * 60)

    print(
        "PROCESO TERMINADO"
    )

    print("=" * 60)

    print(
        f"Productos obtenidos: "
        f"{len(df)}"
    )

    print(
        f"Columnas obtenidas: "
        f"{len(df.columns)}"
    )

    print(
        f"\nArchivo:"
    )

    print(
        OUTPUT_FILE
    )

    print(
        "\nColumnas:"
    )

    print(
        df.columns.tolist()
    )

    print(
        "\nPrimeros registros:"
    )

    print(
        df.head()
    )


# ============================================================
# EJECUCIÓN
# ============================================================

if __name__ == "__main__":

    main()

SCRAPER DE LAPTOPS MERCADO LIBRE

Buscando productos 1 - 50
GET https://api.mercadolibre.com/sites/MLM/search?q=laptops&limit=50&offset=0
Status: 403
Respuesta de error:
{"message":"forbidden","error":"forbidden","status":403,"cause":[]}
❌ No se pudo obtener la página.

❌ No se encontraron productos.


In [9]:
import requests

url = "https://api.mercadolibre.com/sites/MLM/search"

params = {
    "q": "laptops",
    "limit": 5
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("URL:", response.url)
print("Respuesta:")
print(response.text[:1000])

Status: 403
URL: https://api.mercadolibre.com/sites/MLM/search?q=laptops&limit=5
Respuesta:
{"message":"forbidden","error":"forbidden","status":403,"cause":[]}
